# IAN 603 · Module 6
# Joining Data in PostgreSQL

Combine related tables into defensible analytical datasets.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module6.html


## Learning goals

Work through the examples in order. Focus on what each SQL pattern does, what one output row represents, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database
db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script
print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**. You may save the connection settings on your own computer for the next run.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 6

A join is a **row-matching rule** plus a **row-preservation rule**.

| Join | Keeps |
|---|---|
| INNER JOIN | only matched rows |
| LEFT JOIN | every left row plus matches |
| RIGHT JOIN | every right row plus matches |
| FULL OUTER JOIN | rows from both sides |

### Grain first

If one patient has 3 labs and 2 prescriptions, joining both detail tables directly can create:

```text
3 × 2 = 6 rows
```

That may be correct—or accidental multiplication.

Before joining, write:

> **One output row should represent ______.**

Then choose the join and any pre-aggregation needed to preserve that meaning.


# Teaching database

Run the next cell to create or reset the Module 6 teaching data. Re-run it whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 6\n-- Joining Data in PostgreSQL\n-- ============================================================\n-- Companion file for s_module6.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- INNER, LEFT, RIGHT, FULL OUTER JOIN, anti-joins, ON vs WHERE,\n-- USING, self joins, cross joins, cardinality, pre-aggregation,\n-- and join validation.\n--\n-- Main habit: define what one output row should represent before\n-- choosing and writing the join.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS appointment_slots CASCADE;\nDROP TABLE IF EXISTS departments CASCADE;\nDROP TABLE IF EXISTS patient_vitals CASCADE;\nDROP TABLE IF EXISTS prescriptions CASCADE;\nDROP TABLE IF EXISTS lab_results CASCADE;\nDROP TABLE IF EXISTS admissions_stage CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS medical_staff CASCADE;\nDROP TABLE IF EXISTS patients CASCADE;\n\nCREATE TABLE patients (\n    patient_id integer PRIMARY KEY,\n    name varchar(100) NOT NULL,\n    age integer NOT NULL CHECK (age BETWEEN 0 AND 120),\n    gender varchar(20),\n    home_city varchar(60) NOT NULL\n);\n\nCREATE TABLE departments (\n    department varchar(60) PRIMARY KEY,\n    floor_no integer NOT NULL,\n    service_line varchar(80) NOT NULL\n);\n\nCREATE TABLE medical_staff (\n    staff_id integer PRIMARY KEY,\n    name varchar(100) NOT NULL,\n    department varchar(60) REFERENCES departments(department),\n    specialization varchar(80) NOT NULL,\n    hire_date date NOT NULL\n);\n\nCREATE TABLE admissions (\n    admission_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    admit_date date NOT NULL,\n    discharge_date date,\n    diagnosis varchar(120) NOT NULL,\n    department varchar(60) NOT NULL REFERENCES departments(department),\n    attending_staff_id integer REFERENCES medical_staff(staff_id),\n    CHECK (discharge_date IS NULL OR discharge_date >= admit_date)\n);\n\n-- Staging data intentionally has no foreign key so students can see an orphan row.\nCREATE TABLE admissions_stage (\n    import_row_id integer PRIMARY KEY,\n    patient_id integer,\n    diagnosis varchar(120),\n    source_file varchar(80)\n);\n\nCREATE TABLE lab_results (\n    test_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    test_name varchar(60) NOT NULL,\n    result_value numeric(10,2),\n    test_date date NOT NULL,\n    normal_range varchar(30)\n);\n\nCREATE TABLE prescriptions (\n    rx_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    medication varchar(100) NOT NULL,\n    dosage varchar(50) NOT NULL,\n    prescribed_date date NOT NULL,\n    duration_days integer CHECK (duration_days > 0)\n);\n\nCREATE TABLE patient_vitals (\n    vital_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    vital_name varchar(50) NOT NULL,\n    vital_value numeric(10,2),\n    recorded_date date NOT NULL\n);\n\nCREATE TABLE appointment_slots (\n    slot_label varchar(30) PRIMARY KEY\n);\n\nINSERT INTO patients VALUES\n(1001,'Alice Smith',45,'Female','Greensboro'),\n(1002,'Bob Johnson',32,'Male','High Point'),\n(1003,'Carol White',58,'Female','Greensboro'),\n(1004,'David Brown',67,'Male','Burlington'),\n(1005,'Eve Davis',29,'Female','Greensboro'),\n(1006,'Frank Miller',71,'Male','Reidsville'),\n(1007,'Grace Wilson',52,'Female','Greensboro'),\n(1008,'Henry Taylor',48,'Male','High Point'),\n(1009,'Ivy Clark',63,'Female','Greensboro'),\n(1010,'Jack Anderson',55,'Male','Burlington'),\n(1011,'Kara Young',41,'Female','Greensboro');\n\nINSERT INTO departments VALUES\n('Cardiology',3,'Cardiovascular'),\n('Endocrinology',4,'Medicine'),\n('Internal Medicine',2,'Medicine'),\n('Neurology',5,'Neurosciences'),\n('Orthopedics',1,'Musculoskeletal'),\n('Pulmonology',4,'Medicine'),\n('Dermatology',6,'Specialty Care');\n\nINSERT INTO medical_staff VALUES\n(501,'Dr. Sarah Chen','Cardiology','Cardiologist','2020-03-15'),\n(502,'Dr. Michael Rodriguez','Internal Medicine','Internist','2019-07-22'),\n(503,'Dr. Emily Watson','Orthopedics','Orthopedic Surgeon','2021-01-10'),\n(504,'Dr. James Kim','Neurology','Neurologist','2018-11-30'),\n(505,'Dr. Lisa Thompson','Endocrinology','Endocrinologist','2022-05-18'),\n(506,'Dr. Robert Brown','Pulmonology','Pulmonologist','2020-09-05'),\n(507,'Dr. Amanda Garcia','Cardiology','Cardiac Electrophysiologist','2021-08-12');\n\nINSERT INTO admissions VALUES\n(101,1001,'2026-01-15','2026-01-20','Pneumonia','Internal Medicine',502),\n(102,1002,'2026-01-16','2026-01-18','Fractured arm','Orthopedics',503),\n(103,1003,'2026-01-17','2026-01-25','Cardiac arrhythmia','Cardiology',501),\n(104,1001,'2026-03-05','2026-03-12','Hypertension','Internal Medicine',502),\n(105,1004,'2026-02-19','2026-02-26','Heart disease','Cardiology',507),\n(106,1005,'2026-04-20','2026-04-22','Migraine','Neurology',504),\n(107,1006,'2026-05-21','2026-05-28','Diabetes complications','Endocrinology',505),\n(108,1007,'2026-05-22','2026-05-24','Asthma','Pulmonology',506),\n(109,1008,'2026-06-23','2026-06-30','Cardiac monitoring','Cardiology',501),\n(110,1009,'2026-07-24','2026-07-27','Joint pain','Orthopedics',503),\n(111,1010,'2026-07-25',NULL,'Hypertension','Internal Medicine',502),\n(112,1006,'2026-08-01','2026-08-05','Cardiac assessment','Cardiology',507);\n\nINSERT INTO admissions_stage VALUES\n(9001,1001,'Pneumonia','july_import.csv'),\n(9002,1010,'Hypertension','july_import.csv'),\n(9003,1999,'External transfer','july_import.csv');\n\nINSERT INTO lab_results VALUES\n(201,1001,'Glucose',98,'2026-01-16','70-110'),\n(202,1002,'Cholesterol',185,'2026-01-17','<200'),\n(203,1003,'Glucose',145,'2026-01-18','70-110'),\n(204,1004,'Blood Pressure',150,'2026-02-19','<140'),\n(205,1005,'Cholesterol',210,'2026-04-20','<200'),\n(206,1006,'Glucose',320,'2026-05-21','70-110'),\n(207,1007,'Blood Pressure',128,'2026-05-22','<140'),\n(208,1008,'Cholesterol',195,'2026-06-23','<200'),\n(209,1009,'Glucose',110,'2026-07-24','70-110'),\n(210,1010,'Blood Pressure',142,'2026-07-25','<140'),\n(211,1001,'Hemoglobin',14.2,'2026-01-16','12-16'),\n(212,1003,'Hemoglobin',11.8,'2026-01-18','12-16');\n\nINSERT INTO prescriptions VALUES\n(301,1001,'Amoxicillin','500mg','2026-01-16',7),\n(302,1002,'Ibuprofen','400mg','2026-01-17',14),\n(303,1003,'Metoprolol','25mg','2026-01-18',90),\n(304,1004,'Atorvastatin','40mg','2026-02-19',30),\n(305,1006,'Metformin','1000mg','2026-05-21',60),\n(306,1006,'Insulin','50 units','2026-05-21',90),\n(307,1001,'Lisinopril','5mg','2026-03-06',30),\n(308,1007,'Albuterol','2 puffs','2026-05-22',15),\n(309,1008,'Aspirin','81mg','2026-06-23',365),\n(310,1009,'Ibuprofen','400mg','2026-07-24',7);\n\nINSERT INTO patient_vitals VALUES\n(401,1001,'Heart Rate',88,'2026-01-15'),\n(402,1001,'Temperature C',38.4,'2026-01-15'),\n(403,1003,'Heart Rate',104,'2026-01-17'),\n(404,1004,'Systolic BP',150,'2026-02-19'),\n(405,1006,'Glucose',320,'2026-05-21');\n\nINSERT INTO appointment_slots VALUES\n('08:00'),('10:00'),('13:00');"

execute_script(setup_sql)


# RELATIONSHIP / GRAIN GUIDE


patients          : one row per patient
admissions        : one row per admission
lab_results       : one row per lab test
prescriptions     : one row per prescription
medical_staff     : one row per staff member
departments       : one row per department
admissions_stage  : imported rows; may contain unmatched IDs
Joining patients to admissions changes grain from patient
to admission detail when a patient has multiple admissions.


# RUNNABLE MODULE EXAMPLES


## Example 01

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, a.admission_id, a.diagnosis, a.department\nFROM patients AS p\nINNER JOIN admissions AS a\n  ON a.patient_id = p.patient_id\nORDER BY p.patient_id, a.admit_date;"

display_query(query, title="Example 01")


## Example 02

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, a.admission_id, a.diagnosis\nFROM patients AS p\nLEFT JOIN admissions AS a\n  ON a.patient_id = p.patient_id\nORDER BY p.patient_id, a.admission_id;"

display_query(query, title="Example 02")


## Example 03

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name\nFROM patients AS p\nLEFT JOIN lab_results AS l\n  ON l.patient_id = p.patient_id\nWHERE l.test_id IS NULL\nORDER BY p.patient_id;"

display_query(query, title="Example 03")


## Example 04

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT d.department, d.floor_no, s.name AS staff_name, s.specialization\nFROM medical_staff AS s\nRIGHT JOIN departments AS d\n  ON d.department = s.department\nORDER BY d.department, s.name;"

display_query(query, title="Example 04")


## Example 05

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT COALESCE(p.patient_id, s.patient_id) AS patient_id,\n       p.name,\n       s.import_row_id,\n       s.diagnosis,\n       CASE\n         WHEN p.patient_id IS NULL THEN 'Staging row has no patient master'\n         WHEN s.import_row_id IS NULL THEN 'Patient has no staging row'\n         ELSE 'Matched'\n       END AS match_status\nFROM patients AS p\nFULL OUTER JOIN admissions_stage AS s\n  ON s.patient_id = p.patient_id\nORDER BY COALESCE(p.patient_id, s.patient_id);"

display_query(query, title="Example 05")


## Example 06

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, l.test_id, l.result_value\nFROM patients AS p\nLEFT JOIN lab_results AS l\n  ON l.patient_id = p.patient_id\n AND l.test_name = 'Glucose'\nORDER BY p.patient_id;"

display_query(query, title="Example 06")


## Example 07

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, l.test_id, l.result_value\nFROM patients AS p\nLEFT JOIN lab_results AS l\n  ON l.patient_id = p.patient_id\nWHERE l.test_name = 'Glucose'\nORDER BY p.patient_id;"

display_query(query, title="Example 07")


## Example 08

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, p.name, a.admission_id, a.diagnosis\nFROM patients AS p\nJOIN admissions AS a USING (patient_id)\nORDER BY patient_id, a.admission_id;"

display_query(query, title="Example 08")


## Example 09

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT s1.department,\n       s1.name AS staff_member,\n       s2.name AS colleague\nFROM medical_staff AS s1\nJOIN medical_staff AS s2\n  ON s1.department = s2.department\n AND s1.staff_id < s2.staff_id\nORDER BY s1.department, s1.name;"

display_query(query, title="Example 09")


## Example 10

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT a1.patient_id, p.name,\n       a1.admission_id AS earlier_admission,\n       a2.admission_id AS later_admission,\n       a2.admit_date - a1.discharge_date AS days_between\nFROM admissions AS a1\nJOIN admissions AS a2\n  ON a1.patient_id = a2.patient_id\n AND a1.admission_id < a2.admission_id\nJOIN patients AS p\n  ON p.patient_id = a1.patient_id\nWHERE a1.discharge_date IS NOT NULL\n  AND a2.admit_date - a1.discharge_date BETWEEN 1 AND 90\nORDER BY days_between;"

display_query(query, title="Example 10")


## Example 11

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT d.department, s.slot_label\nFROM departments AS d\nCROSS JOIN appointment_slots AS s\nWHERE d.department IN ('Cardiology','Internal Medicine')\nORDER BY d.department, s.slot_label;"

display_query(query, title="Example 11")


## Example 12

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name,\n       COUNT(*) AS joined_rows,\n       COUNT(DISTINCT a.admission_id) AS admissions,\n       COUNT(DISTINCT l.test_id) AS labs,\n       COUNT(DISTINCT pr.rx_id) AS prescriptions\nFROM patients AS p\nLEFT JOIN admissions AS a ON a.patient_id = p.patient_id\nLEFT JOIN lab_results AS l ON l.patient_id = p.patient_id\nLEFT JOIN prescriptions AS pr ON pr.patient_id = p.patient_id\nGROUP BY p.patient_id, p.name\nORDER BY joined_rows DESC, p.patient_id;"

display_query(query, title="Example 12")


## Example 13

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH admission_counts AS (\n  SELECT patient_id, COUNT(*) AS admissions\n  FROM admissions\n  GROUP BY patient_id\n),\nlab_counts AS (\n  SELECT patient_id, COUNT(*) AS labs\n  FROM lab_results\n  GROUP BY patient_id\n),\nrx_counts AS (\n  SELECT patient_id, COUNT(*) AS prescriptions\n  FROM prescriptions\n  GROUP BY patient_id\n)\nSELECT p.patient_id, p.name,\n       COALESCE(a.admissions,0) AS admissions,\n       COALESCE(l.labs,0) AS labs,\n       COALESCE(r.prescriptions,0) AS prescriptions\nFROM patients AS p\nLEFT JOIN admission_counts AS a USING (patient_id)\nLEFT JOIN lab_counts AS l USING (patient_id)\nLEFT JOIN rx_counts AS r USING (patient_id)\nORDER BY p.patient_id;"

display_query(query, title="Example 13")


## Example 14

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.name,\n       a.admission_id,\n       a.diagnosis,\n       a.department,\n       ms.name AS attending_clinician,\n       ms.specialization\nFROM patients AS p\nJOIN admissions AS a\n  ON a.patient_id = p.patient_id\nLEFT JOIN medical_staff AS ms\n  ON ms.staff_id = a.attending_staff_id\nORDER BY a.admit_date, p.name;"

display_query(query, title="Example 14")


## Example 15

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT d.department,\n       COUNT(DISTINCT a.admission_id) AS admissions,\n       COUNT(DISTINCT a.patient_id) AS patients,\n       COUNT(DISTINCT s.staff_id) AS staff_count,\n       ROUND(AVG(a.discharge_date - a.admit_date),1) AS avg_stay_days,\n       string_agg(DISTINCT s.specialization, ', ' ORDER BY s.specialization) AS specializations\nFROM departments AS d\nLEFT JOIN admissions AS a\n  ON a.department = d.department\nLEFT JOIN medical_staff AS s\n  ON s.department = d.department\nGROUP BY d.department\nORDER BY admissions DESC, d.department;"

display_query(query, title="Example 15")


## Example 16

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name,\n       COUNT(*) AS joined_rows,\n       COUNT(a.admission_id) AS naive_admission_count,\n       COUNT(DISTINCT a.admission_id) AS true_admissions\nFROM patients AS p\nLEFT JOIN admissions AS a ON a.patient_id = p.patient_id\nLEFT JOIN lab_results AS l ON l.patient_id = p.patient_id\nLEFT JOIN prescriptions AS pr ON pr.patient_id = p.patient_id\nGROUP BY p.patient_id, p.name\nORDER BY joined_rows DESC, p.patient_id;"

display_query(query, title="Example 16")


## Example 17

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.patient_id, p.name, l.test_id, l.result_value\nFROM patients AS p\nLEFT JOIN lab_results AS l\n  ON l.patient_id = p.patient_id\n AND l.test_name = 'Glucose'\nORDER BY p.patient_id;"

display_query(query, title="Example 17")


## Example 18

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH patient_timeline AS (\n    SELECT patient_id, admit_date AS event_date, 'Admission'::text AS event_type,\n           diagnosis::text AS event_details\n    FROM admissions\n    UNION ALL\n    SELECT patient_id, test_date, 'Lab Test',\n           test_name || ': ' || COALESCE(result_value::text,'NULL')\n    FROM lab_results\n    UNION ALL\n    SELECT patient_id, prescribed_date, 'Prescription',\n           medication || ' - ' || dosage\n    FROM prescriptions\n)\nSELECT p.name, t.event_date, t.event_type, t.event_details\nFROM patient_timeline AS t\nJOIN patients AS p USING (patient_id)\nORDER BY p.name, t.event_date, t.event_type;"

display_query(query, title="Example 18")


## Example 19

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH seq AS (\n    SELECT patient_id, department, admit_date,\n           LEAD(department) OVER (PARTITION BY patient_id ORDER BY admit_date) AS next_department,\n           LEAD(admit_date) OVER (PARTITION BY patient_id ORDER BY admit_date) AS next_admit_date\n    FROM admissions\n)\nSELECT department AS from_department,\n       next_department AS to_department,\n       COUNT(*) AS transitions,\n       ROUND(AVG(next_admit_date - admit_date),1) AS avg_days_between\nFROM seq\nWHERE next_department IS NOT NULL\n  AND department IS DISTINCT FROM next_department\nGROUP BY department, next_department\nORDER BY transitions DESC, from_department, to_department;"

display_query(query, title="Example 19")


## Example 20

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH admission_counts AS (\n  SELECT patient_id, COUNT(*) AS admissions\n  FROM admissions\n  GROUP BY patient_id\n),\nlab_counts AS (\n  SELECT patient_id, COUNT(*) AS labs\n  FROM lab_results\n  GROUP BY patient_id\n),\nrx_counts AS (\n  SELECT patient_id, COUNT(*) AS prescriptions\n  FROM prescriptions\n  GROUP BY patient_id\n)\nSELECT p.patient_id, p.name,\n       COALESCE(a.admissions,0) AS admissions,\n       COALESCE(l.labs,0) AS labs,\n       COALESCE(r.prescriptions,0) AS prescriptions\nFROM patients AS p\nLEFT JOIN admission_counts AS a USING (patient_id)\nLEFT JOIN lab_counts AS l USING (patient_id)\nLEFT JOIN rx_counts AS r USING (patient_id)\nORDER BY p.patient_id;"

display_query(query, title="Example 20")


# JOIN VALIDATION


Compare source admissions with patient-admission INNER JOIN.
SELECT COUNT(*) AS admission_rows
FROM admissions;
SELECT COUNT(*) AS joined_rows
FROM patients AS p
JOIN admissions AS a
  ON a.patient_id = p.patient_id;
Find patient IDs with no admission.
SELECT p.patient_id, p.name
FROM patients AS p
LEFT JOIN admissions AS a
  ON a.patient_id = p.patient_id
WHERE a.admission_id IS NULL
ORDER BY p.patient_id;
Detect staging records without a patient master.
SELECT s.*
FROM admissions_stage AS s
LEFT JOIN patients AS p
  ON p.patient_id = s.patient_id
WHERE p.patient_id IS NULL
ORDER BY s.import_row_id;
Demonstrate row multiplication when multiple child tables are
joined directly at patient level.
SELECT p.patient_id, p.name,
       COUNT(*) AS joined_rows,
       COUNT(DISTINCT a.admission_id) AS admissions,
       COUNT(DISTINCT l.test_id) AS labs,
       COUNT(DISTINCT r.rx_id) AS prescriptions
FROM patients AS p
LEFT JOIN admissions AS a ON a.patient_id = p.patient_id
LEFT JOIN lab_results AS l ON l.patient_id = p.patient_id
LEFT JOIN prescriptions AS r ON r.patient_id = p.patient_id
GROUP BY p.patient_id, p.name
ORDER BY joined_rows DESC, p.patient_id;


# Practice

1. INNER JOIN patients to admissions.
2. LEFT JOIN patients to admissions and preserve all patients.
3. Find patients with no lab result.
4. Keep all patients while matching only Glucose labs.
5. Pre-aggregate admissions, labs, and prescriptions by patient
   before joining the summaries.
6. Summarize admissions and staff counts by department.
CHECKLIST
[ ] I can state the target grain.
[ ] I know which side/population must be preserved.
[ ] I understand one-to-many row expansion.
[ ] I distinguish filtering in ON from filtering in WHERE.
[ ] I validate counts and unmatched rows after joining.

Work on each task before reviewing earlier examples.


In [ ]:
# Write your practice SQL here
practice_query = """

"""

# display_query(practice_query, title="Practice")


# Module 6 summary

Before moving on, make sure you can explain the main patterns in your own words.

For any query, ask:
1. What question does it answer?
2. What does one input row represent?
3. What does one output row represent?
4. What SQL rule changed or filtered the data?
5. How would you validate the result?

## Next
Continue to **Module 7**.
